In [1]:
from pathlib import Path
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Import dataset

with open("../data/surnames_hy.txt", "r", encoding="utf-8") as file:
    surnames = [line.strip() for line in file.readlines()]

surnames[:10]

['աբազյան',
 'աբալակով',
 'աբալյան',
 'աբայան',
 'աբաջյան',
 'աբասյան',
 'աբարդյան',
 'աբարյան',
 'աբարտյան',
 'աբգարյան']

In [3]:
# Convert all surnames to lowercase
surnames = [surname.lower() for surname in surnames]
# Remove duplicates
surnames = list(set(surnames))

surnames[:10]

['խոջամիրյան',
 'ագուլյան',
 'թոքմաջյան',
 'թառանյան',
 'գասապյան',
 'գոմցյան',
 'երեմյան',
 'էդիլյան',
 'զարդարյան',
 'իսոյան']

In [4]:
# Analyze the dataset

total_surnames = len(surnames)
max_surname_length = max(len(surname) for surname in surnames)
min_surname_length = min(len(surname) for surname in surnames)

print(f"Total surnames: {total_surnames}")
print(f"Max surname length: {max_surname_length}")
print(f"Min surname length: {min_surname_length}")


Total surnames: 689
Max surname length: 14
Min surname length: 5


In [5]:
letters_frequency = {}
for surname in surnames:
    for letter in surname:
        if letter in letters_frequency:
            letters_frequency[letter] += 1
        else:
            letters_frequency[letter] = 1

letters_frequency = dict(sorted(letters_frequency.items(), key=lambda item: item[1], reverse=True))

print(f"Letters frequency: {letters_frequency}")

Letters frequency: {'ա': 1511, 'ն': 903, 'յ': 710, 'ր': 284, 'ո': 276, 'ի': 173, 'մ': 167, 'ե': 160, 'ս': 151, 'լ': 149, 'բ': 148, 'ւ': 118, 'դ': 103, 'թ': 98, 'կ': 98, 'գ': 94, 'զ': 83, 'ղ': 70, 'վ': 66, 'տ': 64, 'ք': 51, 'ջ': 46, 'խ': 43, 'շ': 43, 'հ': 38, 'պ': 35, 'չ': 32, 'ց': 29, 'փ': 21, 'ռ': 16, 'ֆ': 14, 'և': 12, 'ճ': 11, 'օ': 7, 'է': 6, 'ժ': 6, 'ծ': 4, 'ձ': 1}


In [6]:
# Check unique letters in the dataset to ensure they all the Armenian alphabet listed in the dataset. If not, we will need to add them to the dataset.

unique_letters = set()
for surname in surnames:
    for letter in surname:
        unique_letters.add(letter)

print(f"Count of unique letters: {len(unique_letters)}")

#sort the unique letters in alphabetical order
unique_letters = sorted(unique_letters)
print(f"Unique letters: {unique_letters}")

Count of unique letters: 38
Unique letters: ['ա', 'բ', 'գ', 'դ', 'ե', 'զ', 'է', 'թ', 'ժ', 'ի', 'լ', 'խ', 'ծ', 'կ', 'հ', 'ձ', 'ղ', 'ճ', 'մ', 'յ', 'ն', 'շ', 'ո', 'չ', 'պ', 'ջ', 'ռ', 'ս', 'վ', 'տ', 'ր', 'ց', 'ւ', 'փ', 'ք', 'օ', 'ֆ', 'և']


Analysis of the unique letter counts shows that one character from the Armenian alphabet is absent from the collected dataset.

Note: Armenian letter "ու" is represented as "ւ".

In [7]:
armenian_letters = [
    "ա", "բ", "գ", "դ", "ե", "զ", "է", "ը", "թ",
    "ժ", "ի", "լ", "խ", "ծ", "կ", "հ", "ձ", "ղ",
    "ճ", "մ", "յ", "ն", "շ", "ո", "չ", "պ", "ջ",
    "ռ", "ս", "վ", "տ", "ր", "ց", "ւ", "փ", "ք",
    "օ", "ֆ", "և"
]

special_tokens = ["<START>", "<END>", "<PAD>"]

vocab = special_tokens + armenian_letters

vocab_size = len(vocab)
vocab_size

42

In [8]:
char_to_id = {char: idx for idx, char in enumerate(vocab)} 
id_to_char = {idx: char for idx, char in enumerate(vocab)}

print(char_to_id["ա"])
print(char_to_id["ս"])

print(id_to_char[char_to_id["ա"]])
print(id_to_char[char_to_id["ս"]])

3
31
ա
ս


In [9]:
surname = "սարգսյան"

encoded = [char_to_id[ch] for ch in surname]

print(encoded)

[31, 3, 34, 5, 31, 23, 3, 24]


In [10]:
decoded = "".join(id_to_char[i] for i in encoded)

print(decoded)

սարգսյան


In [11]:
sequence = ["<START>"] + list(surname) + ["<END>"]

encoded = [char_to_id[ch] for ch in sequence]

print(sequence)
print(encoded)

['<START>', 'ս', 'ա', 'ր', 'գ', 'ս', 'յ', 'ա', 'ն', '<END>']
[0, 31, 3, 34, 5, 31, 23, 3, 24, 1]


In [12]:
import numpy as np

In [13]:
embedding_dim = 16
embedding_scale = 0.01
embedding = np.random.rand(vocab_size, embedding_dim) * embedding_scale

print(embedding.shape)

print(embedding[char_to_id["ա"]])
print(embedding[char_to_id["<PAD>"]])

(42, 16)
[0.00999459 0.0049048  0.00010339 0.00710535 0.00215172 0.00796887
 0.0084023  0.00781535 0.00849537 0.00629828 0.00433502 0.0073207
 0.0099851  0.00987677 0.00354008 0.00843806]
[0.00246116 0.00504312 0.00982096 0.00995506 0.00247098 0.00655179
 0.00479505 0.00807736 0.00938082 0.00042947 0.00899294 0.00416149
 0.009788   0.00541929 0.00350877 0.00487883]


In [14]:
ids = np.array(encoded)

vectors = embedding[ids]

print(ids.shape)
print(vectors.shape)

(10,)
(10, 16)


In [15]:
def pad_sequence(sequence, max_length=max_surname_length + 2):
    pad_token = char_to_id["<PAD>"]
    padded_sequence = sequence + [pad_token] * (max_length - len(sequence))
    return padded_sequence

In [16]:
pad_sequence(encoded)

[0, 31, 3, 34, 5, 31, 23, 3, 24, 1, 2, 2, 2, 2, 2, 2]

In [17]:
def encode_surname(surname):
    sequence = ["<START>"] + list(surname) + ["<END>"]
    encoded = [char_to_id[ch] for ch in sequence]
    return encoded

In [18]:
encode_surname("սարգսյան")

[0, 31, 3, 34, 5, 31, 23, 3, 24, 1]

In [19]:
def prepare_dataset(surnames, max_length=max_surname_length + 2):
    prepared_surnames = []
    for surname in surnames:
        encoded_surname = encode_surname(surname)
        pad_surname = pad_sequence(encoded_surname, max_length)
        prepared_surnames.append(pad_surname)
    return np.array(prepared_surnames)

In [20]:
def create_input_output_sequences(sequence, max_length=max_surname_length + 2):
    X = []
    Y = []
    for i in range(1, len(sequence)):
        input_seq = sequence[:i]
        output_seq = sequence[i]
        pad_input_seq = pad_sequence(input_seq, max_length)
        X.append(pad_input_seq)
        Y.append(output_seq)
    return np.array(X), np.array(Y)

In [22]:
SEED = 42

In [23]:
surnames = np.array(surnames)

np.random.seed(42)
np.random.shuffle(surnames)

split_index = int(len(surnames) * 0.8)

train_surnames = surnames[:split_index]
val_surnames = surnames[split_index:]

print("Train surnames:", len(train_surnames))
print("Validation surnames:", len(val_surnames))

Train surnames: 551
Validation surnames: 138


In [24]:
train_surnames_dataset = []
for surname in train_surnames:
    encoded_surname = encode_surname(surname)
    X, Y = create_input_output_sequences(encoded_surname)
    train_surnames_dataset.extend(zip(X, Y))

validation_surnames_dataset = []
for surname in val_surnames:
    encoded_surname = encode_surname(surname)
    X, Y = create_input_output_sequences(encoded_surname)
    validation_surnames_dataset.extend(zip(X, Y))

In [25]:
X_train = np.array([x for x, y in train_surnames_dataset])
Y_train = np.array([y for x, y in train_surnames_dataset])

X_validation = np.array([x for x, y in validation_surnames_dataset])
Y_validation = np.array([y for x, y in validation_surnames_dataset])

In [26]:
print(X_train.shape)
print(Y_train.shape)

print(X_validation.shape)
print(Y_validation.shape)

(5231, 16)
(5231,)
(1299, 16)
(1299,)


In [27]:
import torch

In [28]:
context_length = X_train.shape[1]
vocab_size = len(vocab)

embedding_dim = 16
hidden_dim = 64

print("context length:", context_length)
print("vocab size:", vocab_size)

context length: 16
vocab size: 42


In [29]:
embedding_scale = 0.01

embedding = (
    torch.randn(vocab_size, embedding_dim)
    * embedding_scale
)

print(embedding.shape)

torch.Size([42, 16])


In [30]:
pad_id = char_to_id["<PAD>"]

print(pad_id)

2


In [31]:
embedding[pad_id] = 0

In [32]:
input_dim = context_length * embedding_dim

W1 = torch.randn(input_dim, hidden_dim) * embedding_scale
b1 = torch.zeros(hidden_dim)

In [33]:
print(W1.shape)
print(b1.shape)

torch.Size([256, 64])
torch.Size([64])


In [34]:
W2 = torch.randn(hidden_dim, vocab_size) * 0.01
b2 = torch.zeros(vocab_size)

In [35]:
print(W2.shape)
print(b2.shape)

torch.Size([64, 42])
torch.Size([42])


In [36]:
x = torch.tensor(X_train[0], dtype=torch.long)
y = torch.tensor(Y_train[0], dtype=torch.long)

In [37]:
print([id_to_char[int(i)] for i in x])
print(id_to_char[int(y)])

['<START>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']
ս


In [38]:
import torch
import torch.nn as nn

In [49]:
class SurnameGeneratorModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_layers,
        context_length,
        dropout_rate,
        activation="relu"
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        layers = []

        input_size = context_length * embedding_dim

        for hidden_size in hidden_layers:

            # Linear layer
            layers.append(
                nn.Linear(input_size, hidden_size)
            )

            # Activation
            if activation == "relu":
                layers.append(nn.ReLU())

            elif activation == "tanh":
                layers.append(nn.Tanh())

            else:
                raise ValueError(
                    f"Unknown activation: {activation}"
                )

            # Dropout
            if dropout_rate > 0:
                layers.append(
                    nn.Dropout(dropout_rate)
                )

            input_size = hidden_size

        # Output layer
        layers.append(
            nn.Linear(input_size, vocab_size)
        )

        self.network = nn.Sequential(*layers)

    def forward(self, x):

        # Embedding
        x = self.embedding(x)

        # Flatten
        x = x.reshape(x.size(0), -1)

        # Hidden layers + output
        logits = self.network(x)

        # LogSoftmax for NLLLoss
        return F.log_softmax(logits, dim=1)

In [40]:
from torch.utils.data import TensorDataset, DataLoader

In [41]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [42]:
def train_model(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_examples = 0

    for batch_X, batch_Y in dataloader:
        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_Y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)
        total_examples += batch_X.size(0)

    return total_loss / total_examples


def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_examples = 0

    with torch.no_grad():
        for batch_X, batch_Y in dataloader:
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            outputs = model(batch_X)
            loss = criterion(outputs, batch_Y)

            total_loss += loss.item() * batch_X.size(0)
            total_examples += batch_X.size(0)

    return total_loss / total_examples

In [50]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train, dtype=torch.long)
Y_train_t = torch.tensor(Y_train, dtype=torch.long)

X_val_t = torch.tensor(X_validation, dtype=torch.long)
Y_val_t = torch.tensor(Y_validation, dtype=torch.long)

train_dataloader = DataLoader(
    TensorDataset(X_train_t, Y_train_t),
    batch_size=32,
    shuffle=True
)

val_dataloader = DataLoader(
    TensorDataset(X_val_t, Y_val_t),
    batch_size=32,
    shuffle=False
)

In [51]:
from itertools import product
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


embedding_dims = [16, 32, 64]

hidden_layer_configs = [
    [128],
    [256, 128],
    [256, 128, 64],
    [256, 128, 64, 32]
]

dropout_rates = [0.1, 0.2, 0.3]

activations = ["relu"]

epochs = 50
learning_rate = 0.01

# Negative Log-Likelihood
criterion = nn.NLLLoss()

results = []

best_overall_loss = float("inf")
best_config = None
best_state = None

context_length = X_train.shape[1]
vocab_size = len(char_to_id)


for embedding_dim, hidden_layers, dropout_rate, activation in product(
    embedding_dims,
    hidden_layer_configs,
    dropout_rates,
    activations
):

    print(
        f"\nTesting: "
        f"embedding={embedding_dim}, "
        f"layers={hidden_layers}, "
        f"dropout={dropout_rate}, "
        f"activation={activation}"
    )

    model = SurnameGeneratorModel(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        hidden_layers=hidden_layers,
        context_length=context_length,
        dropout_rate=dropout_rate,
        activation=activation
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    best_val_loss = float("inf")
    best_epoch = 0
    best_model_state = None

    for epoch in range(epochs):

        train_loss = train_model(
            model,
            train_dataloader,
            optimizer,
            criterion,
            device
        )

        val_loss = evaluate_model(
            model,
            val_dataloader,
            criterion,
            device
        )

        # Save best epoch for this configuration
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

        if (epoch + 1) % 5 == 0:
            print(
                f"Epoch {epoch+1}/{epochs} | "
                f"Train: {train_loss:.4f} | "
                f"Val: {val_loss:.4f}"
            )

    results.append({
        "embedding_dim": embedding_dim,
        "hidden_layers": hidden_layers,
        "dropout_rate": dropout_rate,
        "activation": activation,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch
    })

    print(
        f"Best validation loss: {best_val_loss:.4f} "
        f"at epoch {best_epoch}"
    )

    # Save best configuration overall
    if best_val_loss < best_overall_loss:
        best_overall_loss = best_val_loss

        best_config = {
            "embedding_dim": embedding_dim,
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "activation": activation
        }

        best_state = best_model_state


Testing: embedding=16, layers=[128], dropout=0.1, activation=relu
Epoch 5/50 | Train: 1.7264 | Val: 1.8312
Epoch 10/50 | Train: 1.5791 | Val: 1.9887
Epoch 15/50 | Train: 1.4944 | Val: 1.9739
Epoch 20/50 | Train: 1.4667 | Val: 2.0999
Epoch 25/50 | Train: 1.4101 | Val: 2.3465
Epoch 30/50 | Train: 1.4030 | Val: 2.5124
Epoch 35/50 | Train: 1.3732 | Val: 2.6667
Epoch 40/50 | Train: 1.3378 | Val: 2.7355
Epoch 45/50 | Train: 1.3400 | Val: 2.8228
Epoch 50/50 | Train: 1.3064 | Val: 2.9140
Best validation loss: 1.8312 at epoch 5

Testing: embedding=16, layers=[128], dropout=0.2, activation=relu
Epoch 5/50 | Train: 1.6865 | Val: 1.8726
Epoch 10/50 | Train: 1.5915 | Val: 1.9055
Epoch 15/50 | Train: 1.5173 | Val: 2.0136
Epoch 20/50 | Train: 1.4584 | Val: 2.0946
Epoch 25/50 | Train: 1.4161 | Val: 2.1319
Epoch 30/50 | Train: 1.3931 | Val: 2.3910
Epoch 35/50 | Train: 1.3810 | Val: 2.4006
Epoch 40/50 | Train: 1.3858 | Val: 2.4286
Epoch 45/50 | Train: 1.3291 | Val: 2.6211
Epoch 50/50 | Train: 1.3460 | 

In [52]:
results = sorted(
    results,
    key=lambda result: result["best_val_loss"]
)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. Embedding: {result['embedding_dim']}, "
        f"Layers: {result['hidden_layers']}, "
        f"Dropout: {result['dropout_rate']}, "
        f"Val loss: {result['best_val_loss']:.4f}, "
        f"Epoch: {result['best_epoch']}"
    )

print("\nBest configuration:", best_config)
print("Best validation loss:", best_overall_loss)

1. Embedding: 16, Layers: [256, 128], Dropout: 0.1, Val loss: 1.8017, Epoch: 16
2. Embedding: 16, Layers: [256, 128], Dropout: 0.2, Val loss: 1.8066, Epoch: 17
3. Embedding: 32, Layers: [256, 128, 64], Dropout: 0.1, Val loss: 1.8111, Epoch: 10
4. Embedding: 16, Layers: [256, 128, 64, 32], Dropout: 0.1, Val loss: 1.8210, Epoch: 21
5. Embedding: 16, Layers: [128], Dropout: 0.2, Val loss: 1.8233, Epoch: 6
6. Embedding: 32, Layers: [128], Dropout: 0.3, Val loss: 1.8302, Epoch: 8
7. Embedding: 16, Layers: [128], Dropout: 0.1, Val loss: 1.8312, Epoch: 5
8. Embedding: 32, Layers: [128], Dropout: 0.1, Val loss: 1.8466, Epoch: 9
9. Embedding: 64, Layers: [256, 128, 64], Dropout: 0.1, Val loss: 1.8493, Epoch: 9
10. Embedding: 16, Layers: [256, 128, 64], Dropout: 0.1, Val loss: 1.8563, Epoch: 13
11. Embedding: 32, Layers: [128], Dropout: 0.2, Val loss: 1.8579, Epoch: 11
12. Embedding: 64, Layers: [128], Dropout: 0.1, Val loss: 1.8687, Epoch: 5
13. Embedding: 16, Layers: [128], Dropout: 0.3, Val l

In [53]:
torch.save({
    "model_state_dict": best_state,
    "config": best_config,
    "context_length": context_length,
    "vocab_size": vocab_size,
    "validation_loss": best_overall_loss
}, "best_surname_model.pth")

In [54]:
checkpoint = torch.load(
    "best_surname_model.pth",
    map_location=device,
    weights_only=True
)

config = checkpoint["config"]

best_model = SurnameGeneratorModel(
    vocab_size=checkpoint["vocab_size"],
    embedding_dim=config["embedding_dim"],
    hidden_layers=config["hidden_layers"],
    context_length=checkpoint["context_length"],
    dropout_rate=config["dropout_rate"]
).to(device)

best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

SurnameGeneratorModel(
  (embedding): Embedding(42, 16)
  (network): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=128, out_features=42, bias=True)
  )
)

In [55]:
import torch.nn.functional as F

In [64]:
model = best_model

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50

X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

for epoch in range(num_epochs):
    model.train()

    # Forward
    logits = model(X)

    # Softmax: convert logits to probabilities
    probs = F.softmax(
        logits.view(-1, vocab_size),
        dim=1
    )

    # Select probability of the correct target character
    target_probs = probs[
        torch.arange(Y.numel()),
        Y.view(-1)
    ]

    # Negative Log-Likelihood
    loss = -torch.log(target_probs).mean()

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

Epoch 10/50, Loss: 0.9550
Epoch 20/50, Loss: 0.7896
Epoch 30/50, Loss: 0.6940
Epoch 40/50, Loss: 0.6390
Epoch 50/50, Loss: 0.6526


/var/folders/fv/yzgrhj916vg8plv0w2gwcgkm0000gn/T/ipykernel_18155/1555329428.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.long)
/var/folders/fv/yzgrhj916vg8plv0w2gwcgkm0000gn/T/ipykernel_18155/1555329428.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Y = torch.tensor(Y, dtype=torch.long)


In [57]:
def generate_surname(prefix, max_length=20, temperature=1.0):
    model.eval()

    start_id = char_to_id["<START>"]
    end_id = char_to_id["<END>"]

    # Get model dimensions automatically
    embedding_dim = model.embedding.embedding_dim
    input_features = model.network[0].in_features
    context_size = input_features // embedding_dim

    # Convert prefix to IDs
    prefix_ids = [char_to_id[c] for c in prefix]

    # Initial context
    context = [start_id] + prefix_ids

    generated = prefix

    for _ in range(max_length):

        # Keep exactly context_size tokens
        x = context[-context_size:]

        # Pad if necessary
        if len(x) < context_size:
            x = [start_id] * (context_size - len(x)) + x

        x = torch.tensor([x], dtype=torch.long)

        # Forward
        with torch.no_grad():
            logits = model(x)

        # Temperature + Softmax
        probs = F.softmax(
            logits / temperature,
            dim=-1
        )

        # Sample next character
        next_id = torch.multinomial(
            probs,
            num_samples=1
        ).item()

        # Stop at END
        if next_id == end_id:
            break

        next_char = id_to_char[next_id]

        generated += next_char
        context.append(next_id)

    return generated

In [65]:
generate_surname("մ")

'մն'

In [67]:
for _ in range(10):
    print(generate_surname("մ"))

մն
մն
մն
մն
մն
մն
մն
մն
մն
մն
